# Concreteness analysis — Rassereno

Mean concreteness of each keyword's semantic frame against a random baseline (Methods, *Concreteness
analysis*):
- concreteness scores from Brysbaert et al. (2014), translated into Italian and lemmatised (scores of the
  word forms sharing a lemma averaged; `CONCRETENESS_NORMS` in `src/config.py`); a multi-word expression
  not in the norms gets the mean score of its components;
- frame = the keyword and its neighbours in the **unfiltered** BFMN (the network of every number in the
  paper), built with the same code as `bfmn_rassereno.ipynb`;
- null distribution: means of `N_NULL_SAMPLES` random word lists of the same size (*k*, the frame words
  found in the norms), seeded per (sample, keyword); *z*-test, two-sided, significant at $\alpha = 0.1$
  ($|Z| > 1.6449$);
- effect sizes: Cohen's $|d|$ and Cliff's $|\delta|$ (the direction is the sign of $\bar{x} - \hat{\mu}_0$).

Outputs in `out/analysis/concreteness/`: `concreteness_frames.csv` (every frame, with the words not found
in the norms) and the LaTeX tables of the frames more / less concrete than random.

In [ ]:
import pandas as pd

from config import RESSERENO_ROOT, KEYWORDS_RASSERENO, COMMON_CUES_RASSERENO
from bfmn_building import pipeline_df_building, common_cues_only, build_bfmn_graph, frame_words
from concreteness_analysis import (
    N_NULL_SAMPLES,
    load_concreteness_norms,
    concreteness_table,
    concreteness_latex_tables,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

ALPHA = 0.1
OUT_DIR = RESSERENO_ROOT / "out" / "analysis" / "concreteness"

## Samples

Same samples and groups as `bfmn_rassereno.ipynb` (the high school sample holds the northern and
southern high schools together). **New samples** go in `NEW_SAMPLES` (name &rarr;
dataframe in the formatted wide format: `participant_id`, `cue_word`, `valence_cue_word`,
`association_1..3`, `valence_association_1..3`); they are analysed exactly like the others.

In [ ]:
def read_split(rel_path):
    return pd.read_csv(RESSERENO_ROOT / "out" / "splitted_dfs" / rel_path)


SPLIT_FOLDERS = {
    "high_school_2025": "high_school_2025",
    "physics_2025": "physics_2025",
    "psychology": "psychology_exam_anx_paper",
    "gpt_high_school": "high_school_gpt_oss",
    "gpt_physics": "physics_gpt_oss",
    "gpt_psychology": "psychology_gpt_oss",
}
GROUPS = {"high_anx": "high_anx.csv", "low_anx": "low_anx.csv", "all": "all.csv"}
# physics (humans and GPT-oss) is analysed only as a whole sample (anxiety groups too small)
SAMPLE_GROUPS = {
    sample: ["all"] if "physics" in sample else list(GROUPS) for sample in SPLIT_FOLDERS
}

SAMPLES = {
    f"{sample}_{group}": read_split(f"{folder}/{file}")
    for sample, folder in SPLIT_FOLDERS.items()
    for group, file in GROUPS.items()
    if group in SAMPLE_GROUPS[sample]
}

# samples not in the split files above (high_school_2025 already holds every high schooler, north
# and south): name -> dataframe, e.g. {"new_sample": pd.read_csv(...)}
NEW_SAMPLES = {}
SAMPLES.update(NEW_SAMPLES)

# frames of the keywords that are among the common cues (as spelled in the networks)
KEYWORDS = [
    k for k in KEYWORDS_RASSERENO if k in {c.replace("à", "a") for c in COMMON_CUES_RASSERENO}
]
KEYWORDS

## Semantic frames

Each sample goes through `pipeline_df_building` (unfiltered) on the 18 common cues, as in
`bfmn_rassereno.ipynb`; the frame of a keyword is the keyword and its neighbours in the network (the
vertices of EmoAtlas's `extract_word_from_formamentis`).

In [ ]:
frames = {}
for name, df in SAMPLES.items():
    G, _ = build_bfmn_graph(pipeline_df_building(common_cues_only(df), filter="no"))
    for keyword in KEYWORDS:
        words = frame_words(G, keyword)
        if words:
            frames[(name, keyword)] = words
        else:
            print(f"⚠️ {name}: '{keyword}' has no links, no frame")

## Concreteness against the random baseline

In [ ]:
norms = load_concreteness_norms("ita")
concreteness = concreteness_table(frames, norms, n_samples=N_NULL_SAMPLES)

OUT_DIR.mkdir(parents=True, exist_ok=True)
concreteness.to_csv(OUT_DIR / "concreteness_frames.csv", index=False)

significant = concreteness[concreteness["p_z"] < ALPHA].sort_values("mean_diff")
significant[["sample", "keyword", "k", "mean_diff", "z", "cohen_d", "mean", "cliffs_delta"]].round(2)

## LaTeX tables

Frames significantly more and less concrete than random, with the paper's columns. `LABELS` and
`KEYWORD_LABELS` set the names shown in the tables (samples and keywords not listed keep their name).

In [ ]:
LABELS = {
    "high_school_2025_high_anx": "H-S H-anx",
    "high_school_2025_low_anx": "H-S L-anx",
    "high_school_2025_all": "H-S",
    "physics_2025_all": "Phy",
    "psychology_high_anx": "Psy H-anx",
    "psychology_low_anx": "Psy L-anx",
    "psychology_all": "Psy",
    "gpt_high_school_high_anx": "GPT H-S H-anx",
    "gpt_high_school_low_anx": "GPT H-S L-anx",
    "gpt_high_school_all": "GPT H-S",
    "gpt_physics_all": "GPT Phy",
    "gpt_psychology_high_anx": "GPT Psy H-anx",
    "gpt_psychology_low_anx": "GPT Psy L-anx",
    "gpt_psychology_all": "GPT Psy",
}
KEYWORD_LABELS = {
    "matematica": "Mathematics",
    "statistica": "Statistics",
    "fisica": "Physics",
    "scienza": "Science",
    "esame": "Exam",
    "voto": "Grade",
    "insegnante": "Teacher",
    "scuola": "School",
    "ansia": "Anxiety",
    "benessere": "Wellbeing",
}

concreteness_latex_tables(
    concreteness, OUT_DIR, alpha=ALPHA, labels=LABELS, keyword_labels=KEYWORD_LABELS
)